## Anaysis of the IPL Dataset

In [1]:
import pandas as pd

In [2]:
#load clean dataset

df = pd.read_csv('cleaned_ipl_data.csv', low_memory= False)

In [3]:
#we load csv again means date again come in text
df['date'] = pd.to_datetime(df['date'])

In [4]:
#Collapse ball-by-ball data into one row per match
matches_only= df.drop_duplicates(subset='match_id').copy()


In [5]:
#Which team has won the most matches?
matches_only['match_won_by'].value_counts()

match_won_by
Mumbai Indians                 155
Chennai Super Kings            148
Royal Challengers Bengaluru    143
Kolkata Knight Riders          140
Punjab Kings                   126
Delhi Capitals                 125
Rajasthan Royals               123
Sunrisers Hyderabad            102
Gujarat Titans                  47
Lucknow Super Giants            34
Deccan Chargers                 29
Unknown                         25
Rising Pune Supergiants         15
Gujarat Lions                   13
Pune Warriors                   12
Kochi Tuskers Kerala             6
Name: count, dtype: int64

In [6]:
### Which venue has hosted the most matches?

matches_only['venue'].value_counts()

venue
Wankhede Stadium                                                132
Eden Gardens                                                    107
M Chinnaswamy Stadium                                           104
Arun Jaitley Stadium                                            104
MA Chidambaram Stadium                                           98
Rajiv Gandhi International Stadium                               90
Sawai Mansingh Stadium                                           68
Punjab Cricket Association Stadium, Mohali                       61
Narendra Modi Stadium, Ahmedabad                                 53
Dubai International Cricket Stadium                              46
Dr DY Patil Sports Academy                                       37
Sheikh Zayed Stadium                                             37
Maharashtra Cricket Association Stadium                          35
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium     29
Sharjah Cricket Stadium                   

### does winning the toss affect winning the match

In [7]:
#Exclude no-result matches before win/loss comparisons
decided = matches_only[matches_only['match_won_by'] != 'Unknown'].copy()

In [8]:
#Does winning the toss affect winning the match?
decided['toss_winner_won_match'] = decided['toss_winner'] == decided['match_won_by']
decided['toss_winner_won_match'].value_counts(normalize=True) * 100

toss_winner_won_match
True     51.559934
False    48.440066
Name: proportion, dtype: float64

In [9]:
#Does the toss decision (bat vs field) matter?

decided['toss_decision'].value_counts()

toss_decision
field    810
bat      408
Name: count, dtype: int64

In [10]:
#check the win rate for each decision
decided.groupby('toss_decision')['toss_winner_won_match'].mean() *100


toss_decision
bat      45.343137
field    54.691358
Name: toss_winner_won_match, dtype: float64

In [29]:
matches_only.groupby('year')['toss_decision'].value_counts().unstack()

toss_decision,bat,field
year,,
2008,26,32
2009,35,22
2010,39,21
2011,25,48
2012,37,37
2013,45,31
2014,19,41
2015,25,34
2016,11,49


In [18]:
#Top run-scorers
top_scorers = df.groupby('batter')['runs_batter'].sum().sort_values(ascending=False)
top_scorers.head(10).to_csv('top_scores.csv')

In [13]:
## Team win percentage

matches_played = pd.concat([matches_only['batting_team'], matches_only['bowling_team']]).value_counts()
wins = matches_only['match_won_by'].value_counts()

win_pct = (wins/ matches_played*100).round(1)
summary = pd.DataFrame({'played':matches_played, 'wins':wins, 'win_pct':win_pct})
summary = summary.sort_values('win_pct', ascending =False)
summary
                        

,played,wins,win_pct
Gujarat Titans,77.0,47,61.0
Chennai Super Kings,266.0,148,55.6
Mumbai Indians,291.0,155,53.3
Kolkata Knight Riders,278.0,140,50.4
Rising Pune Supergiants,30.0,15,50.0
Royal Challengers Bengaluru,286.0,143,50.0
Rajasthan Royals,251.0,123,49.0
Sunrisers Hyderabad,211.0,102,48.3
Lucknow Super Giants,72.0,34,47.2
Punjab Kings,278.0,126,45.3


In [17]:
# drop unknown becuase i just want to win predictions

summary = summary.drop('Unknown', errors='ignore')
summary.to_csv('team_win_summary.csv')

## Bowling performance

In [31]:
#most wicket
wickets = df.groupby('bowler')['bowler_wicket'].sum().sort_values(ascending=False)
wickets.head(10).to_csv("top_wickets.csv")
wickets.head(10)

bowler
YS Chahal      233
B Kumar        226
SP Narine      209
PP Chawla      192
JJ Bumrah      190
R Ashwin       187
DJ Bravo       183
RA Jadeja      180
Rashid Khan    179
A Mishra       174
Name: bowler_wicket, dtype: int64

In [33]:
#calculate the econamy rate

bowler_stats = df.groupby('bowler').agg(
    runs_conceded=('runs_bowler', 'sum'),
    balls_bowled=("valid_ball",'sum'),
    wickets=('bowler_wicket', 'sum')
                  
)

bowler_stats['overs_bowled']=bowler_stats['balls_bowled']/6
bowler_stats['economy'] = bowler_stats['runs_conceded'] / bowler_stats['overs_bowled']

# filter: only bowlers with at least 300 overs (roughly 50+ matches worth), otherwise small samples dominate
qualified = bowler_stats[bowler_stats['overs_bowled'] >= 300]
qualified.sort_values('economy').head(10).to_csv("top_economy_csv")
qualified.sort_values('economy').head(10)


,runs_conceded,balls_bowled,wickets,overs_bowled,economy
bowler,,,,,
SP Narine,5273,4660,209,776.666667,6.789270
DW Steyn,2523,2182,97,363.666667,6.937672
Harbhajan Singh,4030,3416,150,569.333333,7.078454
SL Malinga,3366,2827,170,471.166667,7.143969
R Ashwin,5652,4710,187,785.000000,7.200000
Rashid Khan,4297,3543,179,590.500000,7.276884
JJ Bumrah,4469,3653,190,608.833333,7.340268
PP Ojha,2332,1899,89,316.500000,7.368088
A Mishra,4145,3371,174,561.833333,7.377633


In [36]:
#Home city inference win percentage 

appearances = pd.concat([
    matches_only[['batting_team','city']].rename(columns={'batting_team':'team'}),
    matches_only[['bowling_team','city']].rename(columns={'bowling_team':'team'})
])
home_city = appearances.groupby('team')['city'].agg(lambda x: x.value_counts().idxmax())

def build_team_match_rows(df, team_col):
    rows = df[[team_col, 'city', 'match_won_by']].copy()
    rows = rows.rename(columns={team_col: 'team'})
    rows['is_home'] = rows['team'].map(home_city) == rows['city']
    rows['won'] = rows['team'] == rows['match_won_by']
    return rows[['team','is_home','won']]

team_matches = pd.concat([
    build_team_match_rows(matches_only, 'batting_team'),
    build_team_match_rows(matches_only, 'bowling_team')
])

home_away = team_matches.groupby(['team','is_home'])['won'].mean().unstack() * 100
home_away.columns = ['away_win_pct', 'home_win_pct']
home_away = home_away.round(1).sort_values('home_win_pct', ascending=False)

home_away.to_csv('home_away_win_pct.csv')
home_away


,away_win_pct,home_win_pct
team,,
Chennai Super Kings,51.1,65.5
Sunrisers Hyderabad,42.6,60.0
Mumbai Indians,49.1,59.3
Rajasthan Royals,45.4,59.1
Gujarat Titans,63.0,58.1
Kolkata Knight Riders,47.2,55.9
Punjab Kings,43.8,50.8
Royal Challengers Bengaluru,50.5,49.0
Rising Pune Supergiants,52.6,45.5
